In [3]:
import sqlite3
import pandas as pd

In [4]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

#NOTE:
Уточнение по формулировке ТЗ:
«для сопоставления транша и расходной операции, необходимо собрать для транша все расходные операции, сумма которых превышает сумму транша (учитывая превысившую транзакцию)»

Я интерпретировала это так: для транша, не имеющего точного совпадения, нужно взять расходные операции того же клиента и счёта, начиная с даты транша, и последовательно накапливать их сумму до момента, когда она впервые превысит сумму транша, включив в результат все операции, которые участвовали в этом накоплении.

In [5]:
cursor.executescript('''
CREATE TABLE tranches (
    inn TEXT,
    credit_num TEXT,
    account TEXT,
    operation_datetime DATETIME,
    operation_sum REAL,
    doc_id INTEGER
);

CREATE TABLE transactions (
    inn INTEGER,
    account TEXT,
    operation_datetime DATETIME,
    operation_sum REAL,
    ctrg_inn INTEGER,
    ctrg_account TEXT,
    doc_id TEXT
);
''')

In [6]:
cursor.executescript('''
INSERT INTO tranches (inn, credit_num, account, operation_datetime, operation_sum, doc_id) VALUES
('1234567890', 'CREDIT001', '40817810000000000001', '2024-01-01 10:00:00', 1000.00, 1),
('1234567890', 'CREDIT002', '40817810000000000002', '2024-01-05 12:00:00', 1500.00, 2),
('1234567890', 'CREDIT003', '40817810000000000003', '2024-01-10 14:00:00', 2000.00, 3),
('2345678901', 'CREDIT004', '40817810000000000004', '2024-02-15 09:30:00', 3000.00, 4),
('3456789012', 'CREDIT005', '40817810000000000005', '2024-03-20 16:45:00', 5000.00, 5),
('4567890123', 'CREDIT006', '40817810000000000006', '2024-04-25 11:15:00', 7500.00, 6),
('5678901234', 'CREDIT007', '40817810000000000007', '2024-05-30 14:20:00', 10000.00, 7),
('6789012345', 'CREDIT008', '40817810000000000008', '2024-06-10 13:00:00', 12500.00, 8),
('7890123456', 'CREDIT009', '40817810000000000009', '2024-07-15 10:45:00', 15000.00, 9),
('8901234567', 'CREDIT010', '40817810000000000010', '2024-08-20 15:30:00', 20000.00, 10);

INSERT INTO transactions (inn, account, operation_datetime, operation_sum, ctrg_inn, ctrg_account, doc_id) VALUES
(1234567890, '40817810000000000001', '2024-01-02 10:10:00', 900.00, 9876543210, '40817810000000000014', 'T1'),
(2345678901, '40817810000000000004', '2024-02-17 11:20:00', 3500.00, 8765432109, '40817810000000000015', 'T2'),
(1234567890, '40817810000000000003', '2024-01-15 14:05:00', 2500.00, 9876543210, '40817810000000000006', 'T3'),
(2345678901, '40817810000000000004', '2024-02-16 10:10:00', 3200.00, 8765432109, '40817810000000000007', 'T4'),
(7890123456, '40817810000000000009', '2024-07-18 10:15:00', 16000.00, 3210987654, '40817810000000000012', 'T5'),
(1234567890, '40817810000000000002', '2024-01-06 12:05:00', 1500.00, 9876543210, '40817810000000000005', 'T6'),
(5678901234, '40817810000000000007', '2024-06-01 14:40:00', 11000.00, 5432109876, '40817810000000000010', 'T7'),
(6789012345, '40817810000000000008', '2024-06-12 13:50:00', 13000.00, 4321098765, '40817810000000000011', 'T8'),
(3456789012, '40817810000000000005', '2024-03-22 15:20:00', 5500.00, 7654321098, '40817810000000000008', 'T9'),
(8901234567, '40817810000000000010', '2024-08-22 15:25:00', 15000.00, 2109876543, '40817810000000000013', 'T10'),
(1234567890, '40817810000000000001', '2024-01-01 10:05:00', 1000.00, 9876543210, '40817810000000000004', 'T11'),
(4567890123, '40817810000000000006', '2024-04-27 11:30:00', 8000.00, 6543210987, '40817810000000000009', 'T12'),
(8901234567, '40817810000000000010', '2024-08-25 16:30:00', 5800.00, 7654321098, '40817810000000000016', 'T13');
''')


In [7]:
conn.commit()

In [28]:
query = '''
WITH tranches_2024 AS (
SELECT *
FROM tranches
WHERE operation_datetime >= '2024-01-01' AND operation_datetime < '2025-01-01'
),

transactions_2024 AS (
SELECT *
FROM transactions
WHERE operation_datetime >= '2024-01-01' AND operation_datetime < '2025-01-01'
),

first_type AS (
SELECT  t_1.doc_id AS tranche_doc_id,
        t_1.credit_num,
        t_1.inn,
        t_1.account,
        t_1.operation_datetime AS tranche_datetime,
        t_1.operation_sum AS tranche_sum,
        t_2.operation_datetime AS transaction_datetime,
        t_2.operation_sum AS transaction_sum,
        t_2.doc_id AS transaction_doc_id,
        t_2.ctrg_inn,
        t_2.ctrg_account,
        'first_type' AS match_type
FROM tranches_2024 t_1
JOIN transactions_2024 t_2
ON t_1.inn = t_2.inn AND t_1.account = t_2.account
WHERE t_1.operation_sum = t_2.operation_sum
  AND t_2.operation_datetime BETWEEN t_1.operation_datetime AND datetime(t_1.operation_datetime, '+10 days')
),

other_tranches AS (
SELECT t.*
FROM tranches_2024 t
LEFT JOIN first_type
ON first_type.tranche_doc_id = t.doc_id
WHERE first_type.tranche_doc_id IS NULL
),

tranches_running_sum AS(
SELECT  o.doc_id AS tranche_doc_id,
        o.credit_num,
        o.inn AS tranche_inn,
        o.account AS tranche_account,
        o.operation_datetime AS tranche_datetime,
        o.operation_sum AS tranche_sum,
        t_2.operation_datetime AS transaction_datetime,
        t_2.operation_sum AS transaction_sum,
        t_2.doc_id AS transaction_doc_id,
        t_2.ctrg_inn,
        t_2.ctrg_account,
        SUM(t_2.operation_sum) OVER(
          PARTITION BY o.doc_id
          ORDER BY t_2.operation_datetime
          ) AS running_sum,
        ROW_NUMBER() OVER(
          PARTITION BY o.doc_id
          ORDER BY t_2.operation_datetime
        ) AS rn
FROM other_tranches AS o
LEFT JOIN transactions_2024 AS t_2
ON o.inn = t_2.inn AND o.account = t_2.account
WHERE t_2.operation_datetime >= o.operation_datetime
),

first_excess AS (
SELECT tranche_doc_id, MIN(rn) AS first_rn
FROM tranches_running_sum
WHERE tranche_sum < running_sum
GROUP BY tranche_doc_id
),

second_type AS (
SELECT  trs.tranche_doc_id,
        trs.credit_num,
        trs.tranche_inn AS inn,
        trs.tranche_account AS account,
        trs.tranche_datetime,
        trs.tranche_sum,
        trs.transaction_datetime,
        trs.transaction_sum,
        trs.transaction_doc_id,
        trs.ctrg_inn,
        trs.ctrg_account,
        'second_type' AS match_type
FROM tranches_running_sum trs
JOIN first_excess fe
ON trs.tranche_doc_id = fe.tranche_doc_id
WHERE trs.rn <= fe.first_rn
)

SELECT * FROM first_type
UNION ALL
SELECT * from second_type
ORDER BY transaction_datetime;
'''

In [29]:
pd.read_sql_query(query, conn)

,tranche_doc_id,credit_num,inn,account,tranche_datetime,tranche_sum,transaction_datetime,transaction_sum,transaction_doc_id,ctrg_inn,ctrg_account,match_type
0,1,CREDIT001,1234567890,40817810000000000001,2024-01-01 10:00:00,1000.0,2024-01-01 10:05:00,1000.0,T11,9876543210,40817810000000000004,first_type
1,2,CREDIT002,1234567890,40817810000000000002,2024-01-05 12:00:00,1500.0,2024-01-06 12:05:00,1500.0,T6,9876543210,40817810000000000005,first_type
2,3,CREDIT003,1234567890,40817810000000000003,2024-01-10 14:00:00,2000.0,2024-01-15 14:05:00,2500.0,T3,9876543210,40817810000000000006,second_type
3,4,CREDIT004,2345678901,40817810000000000004,2024-02-15 09:30:00,3000.0,2024-02-16 10:10:00,3200.0,T4,8765432109,40817810000000000007,second_type
4,5,CREDIT005,3456789012,40817810000000000005,2024-03-20 16:45:00,5000.0,2024-03-22 15:20:00,5500.0,T9,7654321098,40817810000000000008,second_type
5,6,CREDIT006,4567890123,40817810000000000006,2024-04-25 11:15:00,7500.0,2024-04-27 11:30:00,8000.0,T12,6543210987,40817810000000000009,second_type
6,7,CREDIT007,5678901234,40817810000000000007,2024-05-30 14:20:00,10000.0,2024-06-01 14:40:00,11000.0,T7,5432109876,40817810000000000010,second_type
7,8,CREDIT008,6789012345,40817810000000000008,2024-06-10 13:00:00,12500.0,2024-06-12 13:50:00,13000.0,T8,4321098765,40817810000000000011,second_type
8,9,CREDIT009,7890123456,40817810000000000009,2024-07-15 10:45:00,15000.0,2024-07-18 10:15:00,16000.0,T5,3210987654,40817810000000000012,second_type
9,10,CREDIT010,8901234567,40817810000000000010,2024-08-20 15:30:00,20000.0,2024-08-22 15:25:00,15000.0,T10,2109876543,40817810000000000013,second_type
